## Data Generation

In [3]:
import os
import sys

if not hasattr(sys.modules[__name__], "cwd_changed"):
    os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(__name__))))
    sys.modules[__name__].cwd_changed = True

from tqdm import tqdm
import logging 
logging.getLogger('pgmpy').setLevel(logging.WARNING)
logging.getLogger('pySOT').setLevel(logging.WARNING)


import warnings 
warnings.filterwarnings("ignore")
import pandas as pd
# from utils.graph import dag_to_cpdag
# from metrics.graph import compare_graphs
from synthetic.structure import *
from synthetic.binary import * 
from utils.data import save_csv_with_attrs

### Binary Data

In [2]:
from __future__ import annotations

import os
import hashlib
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd


def generate_binary_data(
    folder: str = "data/datasets/binary_data",
    *,
    # NEW: loop over multiple sizes; if provided, we ignore total_vars and iterate these
    total_vars_grid: Optional[Iterable[int]] = None,
    total_vars: int = 30,
    n_roots: int = 5,  # kept for backward-compat; NOT enforced
    p_noise_grid: Iterable[float] = (0, 0.1, 0.2, 0.3, 0.4, 0.5),
    p_syn_grid: Iterable[float] = (0.0, 0.1, 0.2, 0.3, 0.4, 0.5),
    n_samples_grid: Iterable[int] = (500, 1000, 2000, 5000, 10000),
    seed: Optional[int] = 42,
    start_i: int = 0,
    mapping_filename: str = "grid_mapping.csv",
    skip_invalid: bool = True,
    # NEW: loop over multiple families (all saved into the SAME folder)
    families: Optional[Iterable[str]] = None,   # e.g. ("ba","er","small-world")
    family: str = "er",                          # used only if families is None
    # generator knobs
    max_indeg: int = 2,
    max_outdeg: int = 2,
    root_p: float = 0.5,
    p_copy_flip: float = 0.20,
    sharpness: float = 12.0,
    gate_types: Tuple[str, ...] = ("OR", "AND", "NAND", "NOR"),
    max_structure_attempts: int = 200,
) -> Tuple[pd.DataFrame, Dict[int, Dict[str, Any]]]:
    """
    Generates datasets for multiple (total_vars x family) combinations, and SAVES EVERYTHING INTO ONE FOLDER.

    Saves:
      - Data_Graph_{i}.csv (+ .meta.json via save_csv_with_attrs)
      - Metadata_Graph_{i}.csv
      - grid_mapping.csv columns: i, total_vars, dag_family, p_syn, p_noise, samples
    """
    os.makedirs(folder, exist_ok=True)

    base_seed = 0 if seed is None else int(seed)
    i = int(start_i)

    tv_list = list(total_vars_grid) if total_vars_grid is not None else [int(total_vars)]
    fam_list = list(families) if families is not None else [str(family)]

    def _stable_key(s: str) -> int:
        # deterministic across runs (unlike Python's built-in hash)
        return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:8], 16)

    def _colliders(dag) -> List[Any]:
        return [v for v in dag.nodes() if dag.in_degree(v) == 2]

    mapping_rows: List[Dict[str, Any]] = []
    bundle_by_i: Dict[int, Dict[str, Any]] = {}

    # Global: do we ever need colliders?
    want_any_synergy_global = any(float(x) > 0.0 for x in p_syn_grid)

    for n_nodes in tv_list:
        n_nodes = int(n_nodes)
        if n_nodes < 3:
            if skip_invalid:
                continue
            raise ValueError(f"total_vars must be >=3  got {n_nodes}.")

        for fam in fam_list:
            fam = str(fam)
            fam_key = _stable_key(fam)

            # -----------------------------
            # Sample ONE DAG per (n_nodes, family)
            # -----------------------------
            dag = topo = info = None
            collider_nodes: List[Any] = []
            structure_seed: Optional[int] = None

            for attempt in range(int(max_structure_attempts)):
                structure_seed = base_seed + 12345 + (n_nodes * 10_000) + fam_key + attempt
                dag, topo, info = sample_dag_family_with_caps(
                    n_nodes=n_nodes,
                    family=fam,
                    seed=structure_seed,
                    max_indeg=int(max_indeg),
                    max_outdeg=int(max_outdeg),
                )
                collider_nodes = _colliders(dag)

                # If any p_syn > 0 anywhere, we want at least one collider so p_syn can matter.
                if (not want_any_synergy_global) or (len(collider_nodes) > 0):
                    break

            if dag is None:
                raise RuntimeError("Unexpected: DAG sampling did not return a DAG.")

            if want_any_synergy_global and len(collider_nodes) == 0:
                msg = (
                    f"No colliders found for (family={fam}, total_vars={n_nodes}) after {max_structure_attempts} attempts; "
                    f"cannot apply p_syn>0."
                )
                if skip_invalid:
                    print(msg + " Skipping this (family,size).")
                    continue
                raise ValueError(msg)

            # Keep CPDs dependent ONLY on p_syn (not on noise/samples)
            bundle_by_psyn: Dict[float, Dict[str, Any]] = {}

            for p_syn in p_syn_grid:
                p_syn = float(p_syn)
                if p_syn < 0.0 or p_syn > 1.0:
                    if skip_invalid:
                        continue
                    raise ValueError(f"p_syn must be in [0,1], got {p_syn}.")

                if p_syn not in bundle_by_psyn:
                    psyn_key = int(round(p_syn * 1_000_000))
                    seed_params = base_seed + 2_000 + (n_nodes * 10_000) + fam_key + psyn_key

                    node_cpd, cpd_info = build_node_cpds_from_dag(
                        dag,
                        topo=topo,
                        seed_params=seed_params,
                        root_p=float(root_p),
                        p_copy_flip=float(p_copy_flip),
                        p_syn=float(p_syn),
                        sharpness=float(sharpness),
                        gate_types=tuple(gate_types),
                    )

                    bundle_by_psyn[p_syn] = {
                        "dag": dag,
                        "topo": topo,
                        "info": info,
                        "node_cpd": node_cpd,
                        "cpd_info": cpd_info,
                        "n_colliders": int(len(collider_nodes)),
                        "p_syn_used": float(p_syn),
                        "structure_seed": int(structure_seed) if structure_seed is not None else None,
                        "seed_params": int(seed_params),
                        "dag_family": fam,
                        "total_vars": int(n_nodes),
                    }

                bundle = bundle_by_psyn[p_syn]

                for p_noise in p_noise_grid:
                    p_noise = float(p_noise)

                    for n_samples in n_samples_grid:
                        n_samples = int(n_samples)

                        seed_data = base_seed + 10_000 + i

                        df, dag_str, node_spec, meta = simulate_from_dag_cpds(
                            bundle["dag"],
                            bundle["node_cpd"],
                            n_samples=n_samples,
                            seed_data=seed_data,
                            p_flip_global=p_noise,
                            topo=bundle["topo"],
                        )

                        data_path = os.path.join(folder, f"Data_Graph_{i}.csv")
                        meta_path = os.path.join(folder, f"Metadata_Graph_{i}.csv")

                        # Save data with minimal attrs
                        save_csv_with_attrs(
                            df,
                            data_path,
                            attrs={
                                "i": int(i),
                                "total_vars": int(n_nodes),
                                "dag_family": fam,
                                "p_syn": float(p_syn),
                                "p_noise": float(p_noise),
                                "samples": int(n_samples),
                            },
                        )

                        # Save metadata without attrs
                        if isinstance(meta, pd.DataFrame):
                            meta.to_csv(meta_path, index=False)
                        else:
                            pd.DataFrame([{"meta": repr(meta)}]).to_csv(meta_path, index=False)

                        bundle_by_i[i] = bundle
                        mapping_rows.append(
                            {
                                "i": int(i),
                                "total_vars": int(n_nodes),
                                "dag_family": fam,
                                "p_syn": float(p_syn),
                                "p_noise": float(p_noise),
                                "samples": int(n_samples),
                            }
                        )
                        i += 1

    mapping_df = pd.DataFrame(
        mapping_rows, columns=["i", "total_vars", "dag_family", "p_syn", "p_noise", "samples"]
    )
    mapping_path = os.path.join(folder, mapping_filename)
    mapping_df.to_csv(mapping_path, index=False)

    print(f"Saved {len(mapping_df)} datasets + metadata files to {folder}")
    print(f"Saved mapping: {mapping_path}")

    return mapping_df, bundle_by_i

In [ ]:
# mapping_df, bundle_by_i = generate_binary_data(
#     total_vars_grid=(10, 20, 30),
#     families=("ba", "er", "small-world"),   # Vary density of ER dags
#     p_syn_grid=(0.0, 0.5, 1.0),
#     p_noise_grid=(0, 0.1, 0.2, 0.3, 0.4, 0.5),
#     n_samples_grid=(500, 1000, 2000, 5000, 10000),
#     seed=42,
# )

Saved 810 datasets + metadata files to data/datasets/binary_data
Saved mapping: data/datasets/binary_data/grid_mapping.csv


## Mutual-Information/WMS Synergy grid

In [ ]:
import jointpmf.central_driver_methods as cdm
import numpy as np
import os
import pandas as pd

from joblib import Parallel, delayed

from synthetic.precompute_cpds import load_all_precomputes, select_cpds_for_point
from synthetic.structure import sample_dag_family_with_caps
from utils.data import save_csv_with_attrs
from synthetic.jpmf_helpers import construct_jpmf_bn_from_dag


numvalues = 3
OUT_FOLDER = "data/datasets/jpmf_data"

syn_cutoff_grid = [0.4, 0.5, 0.6, 0.7]
pair_probs_grid = [0.4, 0.5, 0.6, 0.7]

multipliers = [0.20, 0.30, 0.40, 0.50, 0.60]
tgrid = [np.log2(numvalues) * m for m in multipliers]

families = ["er", "ba", "small-world"]

n_one_source = 10
n_two_source = 10

num_vars_list = [10, 20, 30]
n_samples = 5000


def run_grid():

    PRECOMP_DIR = os.path.join("data", "_precomputed_cpds")
    srvs_bank, one_bank, two_bank = load_all_precomputes(PRECOMP_DIR)


    def process_grid_point(i, syn_cutoff, target_mi, pair_probs, family, num_vars):

        print(
            f"\n=== i={i} | fam={family} | syn={syn_cutoff} "
            f"| mi={target_mi:.4f} | pair={pair_probs} ==="
        )

        rng = np.random.default_rng(123 + i)

        # -------------------------------
        # 1. Select CPDs
        # -------------------------------

        precomputed_one, precomputed_two, precomputed_srvs, used_tmi = select_cpds_for_point(
            srvs_bank,
            one_bank,
            two_bank,
            syn_cutoff=syn_cutoff,
            target_mi=target_mi,
            n_one_source=n_one_source,
            n_two_source=n_two_source,
            rng=rng,
        )

        # -------------------------------
        # 2. Generate DAG
        # -------------------------------

        dag, topo, info = sample_dag_family_with_caps(
            n_nodes=num_vars,
            family=family,
            seed=123 + i,
            max_indeg=2,
            max_outdeg=2,
        )

        # -------------------------------
        # 3. Construct JPMF BN from DAG
        # -------------------------------

        bn, appended_edges_weight_df = construct_jpmf_bn_from_dag(
            dag,
            precomputed_one,
            precomputed_two,
            precomputed_srvs,
            numvalues=numvalues,
            prob_pairwise=[pair_probs, 1 - pair_probs],
        )

        # -------------------------------
        # 4. Sample dataset
        # -------------------------------

        samples = bn.generate_samples(n_samples)
        df_samples = pd.DataFrame(samples)

        data_path = os.path.join(OUT_FOLDER, f"Data_Graph_{i}.csv")
        meta_path = os.path.join(OUT_FOLDER, f"Metadata_Graph_{i}.csv")

        save_csv_with_attrs(
            df_samples,
            data_path,
            attrs={
                "i": int(i),
                "family": family,
                "target_mi": float(target_mi),
                "pair_probs": float(pair_probs),
                "syn_cutoff": float(syn_cutoff),
                "num_vars": int(num_vars),
            },
        )

        # metadata
        clean = [list(map(int, comb)) for comb in appended_edges_weight_df["Combs"]]
        appended_edges_weight_df = appended_edges_weight_df.copy()
        appended_edges_weight_df["Combs"] = clean
        appended_edges_weight_df.to_csv(meta_path, index=False)

        # print(f"Wrote dataset {i}")

        return {
            "i": int(i),
            "family": family,
            "syn_cutoff": float(syn_cutoff),
            "target_mi": float(target_mi),
            "pair_probs": float(pair_probs),
            "num_vars": int(num_vars),
        }

    # ---------------------------------
    # Grid definition
    # ---------------------------------

    grid_points = []
    idx = 0

    for family in families:
        for num_vars in num_vars_list:
            for syn_cutoff in syn_cutoff_grid:
                for target_mi in tgrid:
                    for pair_probs in pair_probs_grid:

                        grid_points.append(
                            (idx, syn_cutoff, target_mi, pair_probs, family, num_vars)
                        )

                        idx += 1

    # ---------------------------------
    # Run in parallel
    # ---------------------------------

    mapping_rows = Parallel(n_jobs=-1)(
        delayed(process_grid_point)(*params)
        for params in grid_points
    )

    # ---------------------------------
    # Save grid mapping
    # ---------------------------------

    grid_mapping = pd.DataFrame(
        mapping_rows,
        columns=["i", "family", "syn_cutoff", "target_mi", "pair_probs", "num_vars"],
    )

    grid_mapping.to_csv(
        os.path.join(OUT_FOLDER, "grid_mapping.csv"),
        index=False,
    )

    print(f"\nDone. Wrote {len(mapping_rows)} datasets.")

In [ ]:
# run_grid()